# 01 - Contract Discovery And ABI Coverage

This notebook validates the contract registry baseline, optionally refreshes seed entries,
and inspects ABI coverage for known core contracts.

**Evidence posture**
- Registry rows are local repository artifacts (`data/contracts/registry.csv`).
- ABI files are cached artifacts (`data/contracts/abis/*.json`).
- Live ABI fetches are optional and require BaseScan API availability.


## Workflow

1. Load current registry and inspect known addresses.
2. Optionally reseed baseline registry entries.
3. Optionally fetch missing ABIs from BaseScan.
4. Summarize ABI function/event coverage.


In [ ]:
from pathlib import Path
import sys
import json
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")

cwd = Path.cwd().resolve()
root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "notebooks" / "notebook_utils.py").exists():
        root = candidate
        break
if root is None:
    raise RuntimeError("Could not locate repository root containing notebooks/notebook_utils.py")

if root.as_posix() not in sys.path:
    sys.path.insert(0, root.as_posix())

from notebooks import notebook_utils as nbx
ROOT = nbx.setup_notebook_context()
print(f"Project root: {ROOT}")
print(f"Notebook run started (UTC): {nbx.utc_now_iso()}")


In [ ]:
from src.registry.discover import seed_registry
from src.registry.abi import fetch_abi, cache_abi, load_abi

REGISTRY_PATH = ROOT / "data" / "contracts" / "registry.csv"
RUN_SEED_REGISTRY = False
FETCH_MISSING_ABIS = False

if RUN_SEED_REGISTRY:
    seed_registry()

registry = pd.read_csv(REGISTRY_PATH)
print(f"Loaded registry rows: {len(registry)} from {REGISTRY_PATH}")
display(registry)


In [ ]:
summary_rows = []

for row in registry.to_dict("records"):
    chain = str(row.get("chain", ""))
    address = str(row.get("address", "")).lower()
    label = str(row.get("label", ""))

    abi = load_abi(address)
    abi_source = "cache"

    if abi is None and FETCH_MISSING_ABIS:
        try:
            fetched = fetch_abi(address, chain=chain)
            if fetched:
                cache_abi(address, fetched)
                abi = fetched
                abi_source = "basescan_live"
            else:
                abi_source = "basescan_missing"
        except Exception as exc:  # noqa: BLE001
            abi_source = f"fetch_error: {type(exc).__name__}"

    function_count = sum(1 for item in (abi or []) if item.get("type") == "function")
    event_count = sum(1 for item in (abi or []) if item.get("type") == "event")

    summary_rows.append(
        {
            "label": label,
            "chain": chain,
            "address": address,
            "abi_available": abi is not None,
            "abi_source": abi_source,
            "function_count": function_count,
            "event_count": event_count,
        }
    )

abi_summary = pd.DataFrame(summary_rows).sort_values(["abi_available", "function_count"], ascending=[False, False])
display(abi_summary)


In [ ]:
coverage = (
    abi_summary.groupby("abi_available", dropna=False)
    .size()
    .rename("contracts")
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

axes[0].bar(
    abi_summary["label"],
    abi_summary["function_count"],
    color="#2a9d8f",
)
axes[0].set_title("ABI Function Coverage by Contract")
axes[0].set_ylabel("Function count")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(
    coverage["abi_available"].astype(str),
    coverage["contracts"],
    color=["#e76f51" if v == "False" else "#264653" for v in coverage["abi_available"].astype(str)],
)
axes[1].set_title("ABI Availability")
axes[1].set_xlabel("abi_available")
axes[1].set_ylabel("Contract count")

plt.show()

missing = abi_summary[~abi_summary["abi_available"]]
if not missing.empty:
    display(Markdown("**Missing ABI addresses:**"))
    display(missing[["label", "address", "abi_source"]])
else:
    display(Markdown("All registry addresses currently have cached ABIs."))


## Integrity Notes

- This notebook only verifies the **seed registry + ABI coverage** path.
- Multi-source discovery methods in `src/registry/discover.py` remain scaffold-level (`NotImplementedError`).
- For investor-grade updates, keep this notebook as a transparency layer and avoid claiming automated discovery completeness.
